# Phase 3 — Models

**Purpose:** Fit three logistic regression models to the same data and compare them directly.

1. **No pooling:** Each segment gets an independent intercept. Small segments have high variance.
2. **Full pooling:** Single global intercept. Ignores segment membership entirely.
3. **Partial pooling (target):** Segment intercepts drawn from a shared population distribution with hyperparameters estimated from data.

All models use `cores=1` (required on WSL2/Windows) and non-centred parameterisation for the hierarchical model.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import arviz as az
import pymc as pm

from src.bcs.models import fit_no_pooling, fit_full_pooling, fit_partial_pooling, check_divergences

print("Imports loaded successfully.")

## 1. Load prepared panel

In [ ]:
import pandas as pd

panel = pd.read_parquet("../data/customer_panel.parquet")
print(f"Panel loaded: {len(panel):,} customers, {panel['segment_idx'].nunique()} segments")

# Prepare model inputs
outcome = panel["churned"].values
segment_idx = panel["segment_idx"].values
n_segments = panel["segment_idx"].nunique()
print(f"Outcome shape: {outcome.shape}, Churn rate: {outcome.mean():.1%}")
print(f"Segments: {n_segments}")

## 2. Model A — No pooling

Each segment gets an independent intercept estimated only from its own data.
Small segments have high variance; no information is shared between segments.

$$\text{logit}(p_i) = \alpha_{j[i]}$$
$$\alpha_j \sim \mathcal{N}(0, 10)$$

In [ ]:
model_no_pool, trace_no_pool = fit_no_pooling(outcome, segment_idx, n_segments)
check_divergences(trace_no_pool)

## 3. Model B — Full pooling

Single global intercept. Segment membership is ignored entirely.

$$\text{logit}(p_i) = \alpha$$
$$\alpha \sim \mathcal{N}(0, 10)$$

In [ ]:
model_full, trace_full = fit_full_pooling(outcome)
check_divergences(trace_full)

## 4. Model C — Partial pooling (target model)

Segment intercepts are drawn from a shared population distribution.
The hyperparameters $\mu$ and $\tau$ are estimated from data.

$$\text{logit}(p_i) = \alpha_{j[i]}$$
$$\alpha_j \sim \mathcal{N}(\mu, \tau)$$
$$\mu \sim \mathcal{N}(0, 1)$$
$$\tau \sim \text{HalfNormal}(1)$$

**Non-centred parameterisation:** We use `alpha_offset` (standard normal) multiplied by `tau` rather than sampling `alpha` directly from `Normal(mu, tau)`. This is essential for sampling efficiency when `tau` is small. With centred parameterisation, NUTS develops funnel geometry that causes divergences.

In [ ]:
model_partial, trace_partial = fit_partial_pooling(outcome, segment_idx, n_segments)
check_divergences(trace_partial)

## 5. Model diagnostics

In [ ]:
# Summary of partial pooling model
summary = az.summary(trace_partial, var_names=["mu", "tau"], hdi_prob=0.94)
print(summary.to_string())

In [ ]:
# Trace plot for partial pooling hyperparameters
az.plot_trace(trace_partial, var_names=["mu", "tau"])
plt.tight_layout()
plt.show()
print("Trace plots rendered.")

In [ ]:
# Posterior of tau (between-segment heterogeneity)
az.plot_posterior(trace_partial, var_names=["tau"], hdi_prob=0.94)
plt.tight_layout()
plt.show()
tau_samples = trace_partial.posterior["tau"].values.flatten()
tau_median = np.median(tau_samples)
tau_hdi = az.hdi(trace_partial, var_names=["tau"], hdi_prob=0.94)
print(f"tau posterior median: {tau_median:.3f}")
print(f"tau 94% HDI: [{tau_hdi['tau'].hdi_lower.values[0]:.3f}, {tau_hdi['tau'].hdi_high.values[0]:.3f}]")
if tau_median < 0.5:
    print("Interpretation: modest heterogeneity — partial pooling reduces to near full pooling.")
elif tau_median < 2:
    print("Interpretation: meaningful heterogeneity — segments differ on log-odds scale.")
else:
    print("Interpretation: strong heterogeneity — segments are nearly independent.")

## 6. Save traces

In [ ]:
# Save traces for Phase 4 (results and visualisation)
trace_no_pool.to_netcdf("../data/trace_no_pool.nc")
trace_full.to_netcdf("../data/trace_full.nc")
trace_partial.to_netcdf("../data/trace_partial.nc")
print("All traces saved. Ready for Phase 4 (results).")